# 04: Representational Similarity Analysis (RSA)

**Goal**: Analyze the representational geometry of embeddings across models.

RSA is a label-free method that compares the *structure* of embedding spaces
without requiring ground-truth labels. Key insight: do similar examples cluster
similarly across models?

This notebook:
1. Computes Representational Dissimilarity Matrices (RDMs)
2. Measures pairwise model-to-model correlations
3. Runs odd-one-out ranking task (triplet-based evaluation)
4. Visualizes results

Reference: Kriegeskorte et al. (2008) "Representational Similarity Analysis"
https://doi.org/10.3389/neuro.06.004.2008

## 1. Setup and Load Data

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import os

PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env")

CACHE_DIR = Path(os.getenv("CACHE_DIR", "results/embeddings"))
if not CACHE_DIR.is_absolute():
    CACHE_DIR = PROJECT_ROOT / CACHE_DIR
RESULTS_DIR = PROJECT_ROOT / "results"

np.random.seed(42)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 8)

print(f"Project root: {PROJECT_ROOT}")
print(f"Cache directory: {CACHE_DIR}")
print(f"Available embeddings:")
for f in sorted(CACHE_DIR.glob("*.npy")):
    print(f"  {f.name}")

## 2. Load Cached Embeddings

In [ ]:
from datasets import load_dataset

print("Loading cached embeddings...")
embeddings_dict = {
    "SBERT": np.load(CACHE_DIR / "sbert_vsr_train.npy"),
    "CLIP Text": np.load(CACHE_DIR / "clip_text_vsr_train.npy"),
    "CLIP Image": np.load(CACHE_DIR / "clip_image_vsr_train.npy"),
}

for name, emb in embeddings_dict.items():
    norms = np.linalg.norm(emb, axis=1)
    print(f"  {name:15s}: {emb.shape} (norms: {norms.min():.4f} to {norms.max():.4f})")

print(f"\nLoading VSR dataset...")
vsr = load_dataset("cambridgeltl/vsr_random")
train_data = vsr["train"]
relation_labels = np.array([ex["relation"] for ex in train_data])
binary_labels = np.array([int(ex["label"]) for ex in train_data])

print(f"Dataset: {len(train_data)} examples")
print(f"Relations: {len(np.unique(relation_labels))} unique types")

## 3. Compute RDMs and RSA Correlations

In [ ]:
from src.analysis import rsa_analysis_pairwise, compute_rdm, rsa_correlation, balanced_subsample

print("Computing RDMs and RSA correlations...")
print("(Using balanced subsample N=500 for computational efficiency)")

corr_df, rdm_dict = rsa_analysis_pairwise(
    embeddings_dict,
    relation_labels,
    subsample_n=500
)

print(f"\nRSA Correlation Matrix:")
print(corr_df.round(3))

### Visualize RSA Correlation Heatmap

In [ ]:
# Create heatmap
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_df,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    vmin=0,
    vmax=1,
    square=True,
    cbar_kws={"label": "Spearman correlation"},
    ax=ax,
    linewidths=1
)
ax.set_title("Representational Similarity Analysis (RSA)\nModel-to-Model RDM Correlations", 
            fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures" / "04_rsa_correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 04_rsa_correlation_heatmap.png")

## 4. Key RSA Insights

In [ ]:
print("\n" + "="*70)
print("RSA INTERPRETATION")
print("="*70)

print(f"\nPairwise RSA Correlations (RDM structure similarity):")
for m1 in corr_df.index:
    for m2 in corr_df.columns:
        if m1 < m2:
            print(f"  {m1} ↔ {m2}: {corr_df.loc[m1, m2]:.3f}")

sbert_clip_text = corr_df.loc["SBERT", "CLIP Text"]
sbert_clip_image = corr_df.loc["SBERT", "CLIP Image"]
clip_text_image = corr_df.loc["CLIP Text", "CLIP Image"]

print(f"\nInterpretation:")
if sbert_clip_text < 0.5:
    print(f"  - Low SBERT–CLIP Text correlation ({sbert_clip_text:.3f}): fundamentally different representational geometries")
else:
    print(f"  - Moderate SBERT–CLIP Text correlation ({sbert_clip_text:.3f}): some shared structure")
if clip_text_image < 0.5:
    print(f"  - Low CLIP Text–Image correlation ({clip_text_image:.3f}): text and image encoders structure space very differently")
else:
    print(f"  - High CLIP Text–Image correlation ({clip_text_image:.3f}): text and image encoders share geometric structure")

print("\n" + "="*70)

## 5. Odd-One-Out Ranking Task

In [ ]:
from src.analysis import odd_one_out_ranking

print("Running odd-one-out ranking task...")
print("(Triplets: two captions from same relation, one foil from different relation)")
print("(Task: does the model rank same-relation captions closer together?)")

oddo_results = {}
for model_name, embeddings in embeddings_dict.items():
    print(f"\n{model_name}...")
    result = odd_one_out_ranking(embeddings, relation_labels, n_trials=5000, random_state=42)
    oddo_results[model_name] = result
    print(f"  Accuracy: {result['accuracy']:.1%} ({result['correct']}/{result['n_trials']})")

### Visualize Odd-One-Out Accuracy

In [ ]:
# Bar chart of odd-one-out accuracy
model_names = list(oddo_results.keys())
accuracies = [oddo_results[m]["accuracy"] for m in model_names]
chances = [0.5] * len(model_names)  # Random chance

fig, ax = plt.subplots(figsize=(8, 6))
bar_width = 0.35
x = np.arange(len(model_names))

ax.bar(x - bar_width/2, chances, bar_width, label="Random (50%)", color="lightgray", alpha=0.7)
ax.bar(x + bar_width/2, accuracies, bar_width, label="Model Accuracy", color="steelblue")

ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("Odd-One-Out Ranking Task\n(Do same-relation captions rank closer?)", 
            fontsize=13, fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim([0, 1])
ax.legend()

# Add value labels on bars
for i, acc in enumerate(accuracies):
    ax.text(i + bar_width/2, acc + 0.02, f"{acc:.1%}", ha="center", va="bottom", fontsize=11)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "figures" / "04_oddoneout_accuracy.png", dpi=300, bbox_inches="tight")
plt.show()

print("Saved: 04_oddoneout_accuracy.png")

## 6. Summary and Interpretation

In [ ]:
print("\n" + "="*70)
print("RSA SUMMARY")
print("="*70)

print(f"\n1. REPRESENTATIONAL GEOMETRY:")
print(f"   RSA correlations show how similarly models structure the example space")
for i, m1 in enumerate(model_names):
    for j, m2 in enumerate(model_names):
        if i < j:
            corr = corr_df.loc[m1, m2]
            print(f"   {m1:15s} ↔ {m2:15s}: {corr:.3f}")

print(f"\n2. ODD-ONE-OUT RANKING:")
print(f"   Measures structural understanding (triplet-based, label-free)")
for model_name in model_names:
    acc = oddo_results[model_name]["accuracy"]
    advantage = acc - 0.5
    print(f"   {model_name:15s}: {acc:.1%} ({advantage:+.1%} vs chance)")

print(f"\n3. WHAT THIS TELLS US:")
print(f"   - Low RSA correlation = models restructure space in fundamentally different ways")
print(f"   - High odd-one-out = model understands relation *structure* despite no explicit labels")
print(f"   - Together: RSA is orthogonal to probe-based evaluation (label-free vs label-dependent)")

print(f"\n4. IMPLICATIONS:")
print(f"   - If CLIP has high probing F1 but low RSA with SBERT: spatial info encoded differently")
print(f"   - If odd-one-out accuracy high: structure is robust and relation-aware")
print(f"   - Expected: CLIP > SBERT on both probing and odd-one-out")

print("\n" + "="*70)